# Module 01 — The Runtime and the Toolchain

## Exercise 01.5 — Traceback triage

Five broken functions. For each one, IN THIS ORDER:
  1. Read the code and WRITE DOWN your prediction: which exception type, and
     which line raises it. Do this before running. Predicting first is the
     entire point of the exercise; running first teaches you nothing.
  2. Run it (uncomment the call at the bottom, one at a time).
  3. Read the traceback bottom-up and write down which frame was ACTUALLY at
     fault, which is often not the frame that raised.
  4. Fix it.
Record your answers in the ANSWERS block at the bottom of this file.
The distinction being trained here: the frame that RAISES an exception is often
not the frame that CONTAINS the mistake. In case 2 the mistake is two frames
up. In case 4 it is in the caller's data, not the callee's code. Learning to
scan upward for the first frame you actually wrote is the skill that makes
library tracebacks stop being intimidating.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. What actually happens when you run a script

Python is often called "interpreted", which is true but so compressed that it
misleads. There is a compilation step. It just targets a virtual machine instead
of your CPU.

```text
app.py  (source text)
   |
   |  [1] TOKENIZE + PARSE      -> Abstract Syntax Tree
   v                               (SyntaxError happens HERE, before anything runs)
   AST
   |
   |  [2] COMPILE               -> bytecode: a flat list of VM instructions
   v
   code object  (cached to __pycache__/app.cpython-312.pyc for imported modules)
   |
   |  [3] EVALUATE              -> the eval loop executes bytecode one op at a time
   v
   output / side effects        (NameError, TypeError, etc. happen HERE, at runtime)
```


The crucial split is between stage 1-2 and stage 3.

**Stage 1-2 is static.** It happens before a single line of your program runs.
Only *syntax* is checked. This is why a file with a typo on line 500 will not
run line 1:

```text
print("hello")
def broken(:      # SyntaxError
    pass
```


You never see "hello". The whole file failed to compile.

**Stage 3 is dynamic.** Names, types, attributes, and arguments are all resolved
while the program runs. This is why this file *does* print "hello":

In [ ]:
print("hello")
def broken():
    return undefined_name + 1     # NameError, but only if you CALL it

That difference is the origin of most of Python's productivity and most of its
danger, and it is why Module 17 (typing) and Module 18 (testing) exist. In a
compiled language the compiler finds your typo. In Python, a test or a type
checker finds it, or your user does.

### See it for yourself

In [ ]:
import dis

def add(a, b):
    return a + b

dis.dis(add)

```text
  2           0 RESUME                   0
  3           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 BINARY_OP                0 (+)
             10 RETURN_VALUE
```


That is your function. A stack machine: push `a`, push `b`, apply `+`, return.
You do not need to read bytecode fluently — Module 24 goes deep — but you should
know it exists, because it explains performance questions that are otherwise
mysterious. `dis` is the fastest way to settle an argument about what Python
"really does".

Try these and compare:

In [ ]:
dis.dis(compile("x = 1 + 2", "<s>", "exec"))       # constant folding at compile time
dis.dis(compile("s = 'a' + 'b' + 'c'", "<s>", "exec"))
dis.dis(compile("[x*2 for x in y]", "<s>", "exec"))  # comprehensions have their own scope

---

## Concept 4. Script, module, package: three words people use interchangeably and should not

| Term | Definition | Example |
|---|---|---|
| **Script** | A `.py` file you execute directly | `python app.py` |
| **Module** | A `.py` file that gets imported; becomes a module object | `import util` |
| **Package** | A directory of modules, importable as a unit | `import mypkg.util` |

The same file can be both a script and a module. That is what this idiom is
about, and it is the most misunderstood four lines in Python:

In [ ]:
def main() -> None:
    print("doing the work")

if __name__ == "__main__":
    main()

Every module has a `__name__`. When a file is **imported**, `__name__` is the
module's dotted name (`"util"`, `"mypkg.util"`). When a file is **run
directly**, `__name__` is the string `"__main__"`.

So the guard means: *do this only when I am the entry point, not when someone
imports me.* Without it, importing your module executes your program — which is
exactly what happens the first time someone tries to unit-test a script that
lacks the guard.

### Four ways to run Python, and when each is right

```bash
python app.py              # run a file as a script
python -m mypackage        # run a package's __main__.py
python -m mypackage.mod    # run a module inside a package, as a module
python -c "print(1+1)"     # run a string
python                     # REPL
```

`python -m` is not a stylistic preference. It changes `sys.path` and it changes
`__package__`, which is why relative imports work under `-m` and fail under
`python path/to/file.py`. Module 06 covers this in full; for now, absorb the
rule: **if it lives in a package, run it with `-m`.**

---

## Concept 6. Virtual environments, and why they are not optional

Without a virtual environment, every project on your machine shares one set of
installed packages. Project A needs `pydantic 1.x`, project B needs `2.x`. Only
one can win. That is the entire problem, and it is why the tooling exists.

A virtual environment is a directory containing:

- a symlink or copy of a Python interpreter,
- its own `site-packages` where installs land,
- an `activate` script that puts its `bin/` at the front of your `PATH`.

That is all it is. There is no magic and no global registry.

```bash
python3.12 -m venv .venv       # create
source .venv/bin/activate      # activate  (Windows: .venv\Scripts\activate)
which python                   # -> .../project/.venv/bin/python
pip install requests           # lands in .venv/lib/python3.12/site-packages
deactivate                     # leave
```

Prove to yourself where a package landed:

In [ ]:
import requests
print(requests.__file__)

### `uv`: the modern front end

`uv` (from Astral, the ruff people) is a drop-in replacement for `pip`,
`venv`, `pip-tools`, and `pyenv`, written in Rust. It is typically 10 to 100
times faster, and it resolves dependencies properly.

```bash
uv venv --python 3.12          # create an env, installing 3.12 if needed
uv pip install requests        # pip-compatible interface
uv pip compile requirements.in -o requirements.txt   # lockfile
uv run script.py               # run in the project env without activating
```

Both interfaces are in this course. Use `uv` day to day; understand `pip` and
`venv` because they are what exists on every machine and in every CI image.

### Pinning and lockfiles

```text
requests            # "whatever version" -- irreproducible, avoid outside experiments
requests>=2.31      # a floor -- reasonable for a library you publish
requests==2.31.0    # exact -- correct for an application you deploy
```


The rule: **libraries specify ranges, applications pin exactly.** A library that
pins exactly makes itself uninstallable alongside anything else. An application
that does not pin will one day deploy a version you never tested. Module 30
covers the whole packaging model.

---

## Concept 8. Reading a traceback properly

You will read thousands of these. Read them **bottom-up**.

```text
Traceback (most recent call last):
  File "/app/main.py", line 22, in <module>
    report(load("data.csv"))
  File "/app/report.py", line 9, in report
    avg = total / len(rows)
ZeroDivisionError: division by zero
```


| Line | What it tells you |
|---|---|
| Last line | The exception **type** and **message**. This is *what* went wrong. |
| Frame above it | The exact line that raised. This is *where*. |
| Frames above that | The call chain that got you there, outermost first. |

Two refinements that matter in real life:

**When the error is inside a library**, the last few frames are all library
code. Scan upward for the last frame that is *your* file. That is where your
mistake is; the library is usually just reporting it.

**When you see two tracebacks joined by text**, read the phrasing:

- `During handling of the above exception, another exception occurred` — you
  raised a new error *inside* an `except` block. Usually the second one is
  masking the first; the first is the real story.
- `The above exception was the direct cause of the following exception` — someone
  wrote `raise NewError(...) from original`. That is deliberate, good practice,
  and Module 16 teaches it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Python is a specification. CPython is a program.
- Section 2: What actually happens when you run a script
- Section 3: `__pycache__` and the `.pyc` question
- Section 4: Script, module, package: three words people use interchangeably and should not
- Section 5: `sys.path`: how `import` finds things
- Section 6: Virtual environments, and why they are not optional
- Section 7: The REPL is a laboratory
- Section 8: Reading a traceback properly
- Section 9: Why "Python is slow" is an imprecise claim
- Section 10: The tools you will run every day

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations


# --- case 1 -----------------------------------------------------------------

---

## `average`

_average_

In [ ]:
def average(numbers: list[float]) -> float:
    return sum(numbers) / len(numbers)

---

## `case1`

_case1_

In [ ]:
def case1() -> float:
    readings: list[float] = []
    return average(readings)

---

## `parse_row`

_parse row_

In [ ]:
def parse_row(row: str) -> dict[str, str]:
    name, age, city = row.split(",")
    return {"name": name, "age": age, "city": city}

---

## `load`

_load_

In [ ]:
def load(rows: list[str]) -> list[dict[str, str]]:
    return [parse_row(r) for r in rows]

---

## `case2`

_case2_

In [ ]:
def case2() -> list[dict[str, str]]:
    return load(["ada,36,london", "grace,45", "alan,41,cambridge"])

---

## `get_setting`

_get setting_

In [ ]:
def get_setting(config: dict[str, object], key: str) -> object:
    return config[key]

---

## `case3`

_case3_

In [ ]:
def case3() -> object:
    config = {"host": "localhost", "port": 8080}
    return get_setting(config, "timeout")

---

## `total_price`

_total price_

In [ ]:
def total_price(items: list[dict[str, object]]) -> float:
    return sum(item["price"] * item["qty"] for item in items)  # type: ignore[operator]

---

## `case4`

_case4_

In [ ]:
def case4() -> float:
    cart = [
        {"price": 9.99, "qty": 2},
        {"price": "19.99", "qty": 1},  # came from a form, never converted
    ]
    return total_price(cart)

---

## `fetch`

_fetch_

In [ ]:
def fetch(url: str) -> str:
    raise ConnectionError(f"could not reach {url}")

---

## `fetch_with_fallback`

_fetch with fallback_

In [ ]:
def fetch_with_fallback(url: str) -> str:
    try:
        return fetch(url)
    except ConnectionError:
        return fetch(url.replace("https", "http"))

---

## `case5`

_case5_

In [ ]:
def case5() -> str:
    return fetch_with_fallback("https://example.invalid/data")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    # Uncomment ONE at a time, after writing your prediction.
    # print(case1())
    # print(case2())
    # print(case3())
    # print(case4())
    # print(case5())
    pass


ANSWERS = """
case 1
  predicted exception :
  predicted line      :
  actual              :
  frame at fault      :
  fix                 :

case 2
  predicted exception :
  predicted line      :
  actual              :
  frame at fault      :
  fix                 :
  NOTE: which ROW of the input caused it? How did you know from the traceback,
        and what would you add to the code so the traceback tells you next time?

case 3
  predicted exception :
  actual              :
  fix                 :
  NOTE: name two different correct fixes and say when each is right.

case 4
  predicted exception :
  actual              :
  frame at fault      :
  NOTE: the exception is raised inside total_price, but the MISTAKE is in
        case4. Explain the difference in one sentence.

case 5
  predicted exception :
  actual              :
  NOTE: the traceback shows TWO exceptions joined by a sentence. Quote that
        sentence and explain what it means. Which of the two is the real story
        for someone debugging this?
"""

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.